# TorchRL_MAC API Demonstration

This notebook demonstrates the two-layer API for multi-agent cooperation without training.

1. **Native API**: Direct use of `src/envs`, `src/wrappers`, `src/agent_policy`
2. **Wrapper Layer**: High-level helpers from `TorchRL_MAC_utils`

**Note:** This notebook shows API usage and config options. No A3C training is performed here. See `TorchRL_MAC.example.ipynb` for training.

## 1. Imports

In [ ]:
import torch
from src.envs.mpe_env import make_mpe_env
from src.wrappers.wrapper import MPEWrapper
from src.agent_policy.agent_policy import SharedMLPPolicy
from TorchRL_MAC_utils import get_device

print(f"PyTorch version: {torch.__version__}")
print(f"Available device: {get_device()}")

PyTorch version: 2.7.0


## 2. Configuration

Show available config options for environment and A3C training.

In [ ]:
from TorchRL_MAC_utils import EnvConfig, TrainConfig

# Environment config (device=None for auto-detect)
env_cfg = EnvConfig(seed=42, max_steps=25, device=None)
print(f"EnvConfig:")
print(f"  env_name: {env_cfg.env_name}")
print(f"  seed: {env_cfg.seed}")
print(f"  max_steps: {env_cfg.max_steps}")
print(f"  device: {env_cfg.device}")

# Training config with A3C parameters
train_cfg = TrainConfig(
    gamma=0.99,
    lr=3e-4,
    entropy_coef=0.005,
    value_coef=0.5,
    max_grad_norm=0.5,
    max_episodes=300,    # Total episodes across all workers
    log_every=25,
    device="mps",        # Global networks device (workers always use CPU)
    num_workers=4,       # A3C: number of parallel workers
    n_steps=10,          # A3C: steps per worker rollout
    sync_every=1,        # Sync weights after each worker update
    seed=0
)
print(f"\nTrainConfig:")
print(f"  gamma: {train_cfg.gamma}")
print(f"  lr: {train_cfg.lr}")
print(f"  entropy_coef: {train_cfg.entropy_coef}")
print(f"  max_episodes: {train_cfg.max_episodes}")
print(f"  device: {train_cfg.device}")

print(f"\nA3C-specific fields:")
print(f"  num_workers: {train_cfg.num_workers} (parallel worker processes)")
print(f"  n_steps: {train_cfg.n_steps} (steps per worker rollout)")
print(f"  sync_every: {train_cfg.sync_every} (weight sync frequency)")
print(f"\nNote: Workers always use CPU for environment interaction.")
print(f"Global networks use '{train_cfg.device}' for gradient computation.")

EnvConfig: EnvConfig(env_name='simple_spread', seed=42, max_steps=25, device='cpu')


## 3. Native API: Environment and Wrapper

Direct use of `src/envs/mpe_env.py` and `src/wrappers/wrapper.py`

**Note:** Wrapper done semantics: episode ends when ALL agents are done/truncated.

In [3]:
# Create raw PettingZoo env
raw_env = make_mpe_env(N=3, max_cycles=25)
obs_dict, info = raw_env.reset(seed=env_cfg.seed)

print(f"Agents: {raw_env.agents}")
print(f"Agent 0 obs shape: {len(obs_dict[raw_env.agents[0]])}")
print(f"Agent 0 action space: {raw_env.action_space(raw_env.agents[0])}")

# Create wrapper for tensor I/O
wrapper = MPEWrapper(device=env_cfg.device, N=3, max_cycles=env_cfg.max_steps)
obs_tensor = wrapper.reset(seed=env_cfg.seed)

print(f"\nWrapper obs shape: {obs_tensor.shape}")  # [num_agents, obs_dim]
print(f"Number of agents: {wrapper.num_agents}")
print(f"Observation dim: {wrapper.obs_dim}")
print(f"Action count: {wrapper.n_actions}")

# Take one step with random valid actions
random_actions = torch.randint(0, wrapper.n_actions, (wrapper.num_agents,), dtype=torch.long)
next_obs, rewards, done_all = wrapper.step(random_actions)

print(f"\nAfter step:")
print(f"  Next obs shape: {next_obs.shape}")
print(f"  Rewards shape: {rewards.shape}")
print(f"  Done (all): {done_all}")

raw_env.close()
wrapper.env.close()

Agents: ['agent_0', 'agent_1', 'agent_2']
Agent 0 obs shape: 18
Agent 0 action space: Discrete(5)

Wrapper obs shape: torch.Size([3, 18])
Number of agents: 3
Observation dim: 18
Action count: 5

After step:
  Next obs shape: torch.Size([3, 18])
  Rewards shape: torch.Size([3])
  Done (all): False


/Users/ineshtandon/Documents/GitHub/umd_classes/class_project/MSML610/Fall2025/Projects/UmdTask21_Fall2025_TorchRL_Multi-Agent_Cooperation/src/wrappers/wrapper.py:53: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:257.)
  return torch.tensor(obs_list, dtype=torch.float32, device=self.device)


## 4. Wrapper Layer: High-Level Utilities

Using `TorchRL_MAC_utils` to build actor, critic, and run forward passes

In [ ]:
from TorchRL_MAC_utils import (
    make_env,
    build_shared_actor,
    build_central_critic,
)

# Create env and infer dimensions
env = make_env(env_cfg)
obs = env.reset(seed=env_cfg.seed)
n_agents, obs_dim = obs.shape
n_actions = env.n_actions
joint_obs_dim = n_agents * obs_dim

print(f"Environment dimensions:")
print(f"  n_agents: {n_agents}")
print(f"  obs_dim: {obs_dim}")
print(f"  n_actions: {n_actions}")
print(f"  joint_obs_dim: {joint_obs_dim}")

# Build actor and critic networks
actor = build_shared_actor(obs_dim, n_actions, hidden_sizes=(128, 128))
critic = build_central_critic(joint_obs_dim, hidden_sizes=(128, 128))

print(f"\nActor network (parameter-shared across agents):")
print(f"  Input: [n_agents, {obs_dim}]")
print(f"  Output: [n_agents, {n_actions}] (logits)")
print(f"  Parameters: {sum(p.numel() for p in actor.parameters()):,}")

print(f"\nCritic network (centralized, sees joint observation):")
print(f"  Input: [{joint_obs_dim}] (flattened obs)")
print(f"  Output: scalar (value)")
print(f"  Parameters: {sum(p.numel() for p in critic.parameters()):,}")

env.env.close()

Environment dimensions:
  n_agents: 3
  obs_dim: 18
  n_actions: 5
  joint_obs_dim: 54

Actor: SharedMLPPolicy(
  (net): Sequential(
    (0): Linear(in_features=18, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=5, bias=True)
  )
)
Critic: CentralCritic(
  (net): Sequential(
    (0): Linear(in_features=54, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=1, bias=True)
  )
)


### Forward Pass and Action Selection

In [ ]:
# Forward passes with fake tensors (no environment needed)
print("Forward pass examples (using fake tensors):\n")

# Create fake observation tensor
fake_obs = torch.randn(n_agents, obs_dim)
print(f"Fake observation: shape {fake_obs.shape}")

# Actor forward pass
with torch.no_grad():
    logits = actor(fake_obs)
    print(f"\nActor forward pass:")
    print(f"  Input shape: {fake_obs.shape}")
    print(f"  Output (logits) shape: {logits.shape}")
    print(f"  Logits (first agent): {logits[0].numpy()[:4]}... (showing first 4 values)")

# Critic forward pass
with torch.no_grad():
    joint_obs = fake_obs.reshape(-1)  # Flatten observations
    value = critic.value(joint_obs)
    print(f"\nCritic forward pass:")
    print(f"  Joint obs shape: {joint_obs.shape}")
    print(f"  Value (scalar): {value.item():.4f}")

# Show shapes for gradient computation
print(f"\nFor A3C training:")
print(f"  Actor output shape matches [n_agents={n_agents}, n_actions={n_actions}]")
print(f"  Critic value is scalar (for advantage computation)")
print(f"  Both networks use fake data here; actual training uses real env observations")

Test observation shape: torch.Size([3, 18])
Actor logits shape: torch.Size([3, 5])

Sampled actions: tensor([0, 3, 0])
  Shape: torch.Size([3])
  Dtype: torch.int64
Mean log-prob (scalar): -1.5257
Mean entropy (scalar): 1.6064

Critic value (scalar): 0.0772
  Joint obs shape: torch.Size([54])


## Summary

This notebook demonstrates the A3C API **without training**:

1. **TrainConfig with A3C fields:**
   - `num_workers=4` — Number of parallel asynchronous workers
   - `n_steps=10` — Steps per worker rollout before gradient update
   - `max_episodes=300` — Total episodes across all workers
   - `device="mps"` — Global network device (workers always use CPU)
   - `sync_every=1` — How often to sync weights from global to workers

2. **Device Strategy:**
   - Specified device (e.g., `"mps"`) applies to **global networks** for gradient computation
   - Worker processes **always use CPU** for environment interaction and local networks
   - This avoids MPS incompatibility with `torch.multiprocessing` shared memory

3. **Network Architecture:**
   - **Actor:** Parameter-shared across agents, takes local obs `[n_agents, obs_dim]` → logits `[n_agents, n_actions]`
   - **Critic:** Centralized, takes joint obs `[n_agents * obs_dim]` → scalar value

4. **Forward Pass:**
   - Actor: Maps observations to action logits (used for action sampling in workers)
   - Critic: Maps joint observations to value estimate (used for advantage computation)

**Key Insight:** A3C training spawns multiple worker processes that independently collect rollouts and update the global networks. This notebook shows the static parts (configuration, network creation, forward passes). For actual training, see `TorchRL_MAC.example.ipynb`.